# Expense Tracker — Kaggle Master Training

**Run the notebook from top to bottom.** This notebook trains the initial **classical-ML Expense Intelligence pipeline** on Kaggle without modifying Kaggle's system scientific packages.

The initial training run deliberately **does not train a transformer**. Transformer support remains optional in the repository for a later experiment only.

Current ML components:
- **Transaction category classifier:** word + character TF-IDF with Logistic Regression
- **Merchant similarity index:** character TF-IDF + nearest-neighbor retrieval for merchant normalization
- **Duplicate similarity index:** character TF-IDF + nearest-neighbor retrieval for likely duplicate transactions
- **Transaction anomaly model:** Isolation Forest, only when numeric transaction features such as amount are available
- **Spending forecast model:** HistGradientBoostingRegressor, only when date/timestamp and amount features are available

With the currently configured text classification datasets, the first three components are applicable; anomaly detection and spending forecasting are expected to report `not_applicable` unless those datasets provide the required numeric/date fields.


In [1]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Yoge-2004/expense-tracker.git"
BRANCH = "feature/ml-expense-intelligence"
WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "expense-tracker"
ML = REPO / "ml"

OUTPUT = WORK_ROOT / "expense-ml-runs"
DATA_CACHE = WORK_ROOT / "expense-ml-data"
HF_CACHE = Path("/kaggle/temp/huggingface")
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA_CACHE.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["TRANSFORMERS_CACHE"] = str(HF_CACHE / "transformers")
os.environ["MPLBACKEND"] = "Agg"
os.environ["EXPENSE_ML_OUTPUT"] = str(OUTPUT)
os.environ["EXPENSE_ML_DATA_DIR"] = str(DATA_CACHE)
os.environ["EXPENSE_ML_DATA_CACHE"] = str(DATA_CACHE)

def run(*args, cwd=None, env=None):
    merged = os.environ.copy()
    if env:
        merged.update({str(k): str(v) for k, v in env.items()})
    print("$", " ".join(map(str, args)))
    return subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, env=merged, check=True)

if REPO.exists():
    shutil.rmtree(REPO)
run("git", "clone", "--depth", "1", "--branch", BRANCH, "--single-branch", REPO_URL, REPO)
commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Checked-out commit:", commit)
print("ML directory:", ML)

$ git clone --depth 1 --branch feature/ml-expense-intelligence --single-branch https://github.com/Yoge-2004/expense-tracker.git /kaggle/working/expense-tracker


Cloning into '/kaggle/working/expense-tracker'...


Checked-out commit: 1f30641bcfe6ae493f3a32b9781c06649c110071
ML directory: /kaggle/working/expense-tracker/ml


## 1. Prepare Kaggle authentication

This initial classical-ML pipeline does **not require a GPU**. A Kaggle GPU may be enabled, but it is not used for the initial classifier or auxiliary models.

Create a Kaggle Secret named `HF_TOKEN` when the configured Hugging Face datasets require authentication. The token is used only for dataset access and is never printed.


In [2]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token.strip()
except Exception as exc:
    print("HF_TOKEN secret lookup skipped:", type(exc).__name__)

print("HF_TOKEN loaded:", bool(os.environ.get("HF_TOKEN")))


HF_TOKEN loaded: True


## 2. Create the isolated Python 3.14 classical-ML environment

Kaggle's notebook kernel can remain on Python 3.12. The ML project is installed into its own `uv` environment using the Python version declared by the repository.

This initial run installs only the **classical training dependencies**. It does **not** install PyTorch, Transformers, Accelerate, or other transformer-training dependencies.


In [3]:
if shutil.which("uv") is None:
    run(sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "uv")

UV = shutil.which("uv")
if UV is None:
    candidates = [Path(sys.prefix) / "bin" / "uv", Path.home() / ".local" / "bin" / "uv"]
    UV = next((str(p) for p in candidates if p.exists()), None)
if UV is None:
    raise FileNotFoundError("uv could not be located after installation.")

UV_CACHE = WORK_ROOT / "uv-cache"
UV_CACHE.mkdir(parents=True, exist_ok=True)

run(UV, "python", "install", "3.14")
run(UV, "sync", "--extra", "classical", "--extra", "dev", cwd=ML, env={"UV_CACHE_DIR": str(UV_CACHE)})

version = subprocess.check_output([UV, "run", "python", "--version"], cwd=ML, text=True).strip()
print("Project interpreter:", version)
if "3.14" not in version:
    raise RuntimeError(f"uv selected the wrong Python interpreter: {version}")

$ /usr/local/bin/uv python install 3.14


 Downloaded cpython-3.14.5-linux-x86_64-gnu (download)
Installed Python 3.14.5 in 1.56s
 + cpython-3.14.5-linux-x86_64-gnu (python3.14)


$ /usr/local/bin/uv sync --extra classical --extra dev


Using CPython 3.14.5
Creating virtual environment at: .venv
Resolved 107 packages in 1.60s
   Building expense-tracker-ml @ file:///kaggle/working/expense-tracker/ml
 Downloaded kiwisolver
 Downloaded aiohttp
 Downloaded pygments
 Downloaded hf-xet
 Downloaded fonttools
 Downloaded pillow
 Downloaded scikit-learn
 Downloaded ruff
 Downloaded matplotlib
 Downloaded numpy
      Built expense-tracker-ml @ file:///kaggle/working/expense-tracker/ml
 Downloaded scipy
 Downloaded pandas


Project interpreter: Python 3.14.5


 Downloaded pyarrow
Prepared 56 packages in 4.15s
Installed 56 packages in 82ms
 + aiohappyeyeballs==2.7.1
 + aiohttp==3.14.3
 + aiosignal==1.4.0
 + anyio==4.15.1
 + attrs==26.1.0
 + certifi==2026.7.22
 + charset-normalizer==3.5.1
 + click==8.5.0
 + cloudpickle==3.1.2
 + contourpy==1.4.0
 + cycler==0.12.1
 + datasets==5.0.1
 + dill==0.4.1
 + expense-tracker-ml==0.2.0 (from file:///kaggle/working/expense-tracker/ml)
 + filelock==4.0.0
 + fonttools==4.65.0
 + frozenlist==1.8.0
 + fsspec==2026.6.0
 + h11==0.16.0
 + hf-xet==1.6.0
 + httpcore==1.0.9
 + httpx==0.28.1
 + huggingface-hub==1.32.0
 + idna==3.20
 + iniconfig==2.3.0
 + joblib==1.6.0
 + kiwisolver==1.5.1
 + matplotlib==3.11.2
 + multidict==6.8.0
 + multiprocess==0.70.19
 + narwhals==2.26.0
 + numpy==2.5.3
 + packaging==26.3
 + pandas==2.3.3
 + pillow==12.3.0
 + pluggy==1.6.0
 + propcache==0.5.4
 + pyarrow==25.0.1
 + pygments==2.21.0
 + pyparsing==3.3.2
 + pytest==8.4.2
 + python-dateutil==2.9.0.post0
 + pytz==2026.3.post1
 + pyyaml

In [4]:
preflight = r'''
import sys
import numpy, pandas, scipy, sklearn, datasets, matplotlib, tqdm

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Datasets:", datasets.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Matplotlib backend:", matplotlib.get_backend())
print("tqdm:", tqdm.__version__)
'''
run(
    UV, "run", "python", "-c", preflight,
    cwd=ML,
    env={
        "HF_HOME": os.environ["HF_HOME"],
        "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
        "TRANSFORMERS_CACHE": os.environ["TRANSFORMERS_CACHE"],
        "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
        "MPLBACKEND": "Agg",
    },
)


$ /usr/local/bin/uv run python -c 
import sys
import numpy, pandas, scipy, sklearn, datasets, matplotlib, tqdm

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("Pandas:", pandas.__version__)
print("SciPy:", scipy.__version__)
print("scikit-learn:", sklearn.__version__)
print("Datasets:", datasets.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Matplotlib backend:", matplotlib.get_backend())
print("tqdm:", tqdm.__version__)

Python: 3.14.5 (main, May 10 2026, 19:28:16) [Clang 22.1.3 ]
NumPy: 2.5.3
Pandas: 2.3.3
SciPy: 1.18.1
scikit-learn: 1.9.1
Datasets: 5.0.1
Matplotlib: 3.11.2
Matplotlib backend: Agg
tqdm: 4.70.1


CompletedProcess(args=['/usr/local/bin/uv', 'run', 'python', '-c', '\nimport sys\nimport numpy, pandas, scipy, sklearn, datasets, matplotlib, tqdm\n\nprint("Python:", sys.version)\nprint("NumPy:", numpy.__version__)\nprint("Pandas:", pandas.__version__)\nprint("SciPy:", scipy.__version__)\nprint("scikit-learn:", sklearn.__version__)\nprint("Datasets:", datasets.__version__)\nprint("Matplotlib:", matplotlib.__version__)\nprint("Matplotlib backend:", matplotlib.get_backend())\nprint("tqdm:", tqdm.__version__)\n'], returncode=0)

## 3. Start from a completely fresh training dataset/cache

The initial run must not silently reuse prepared data or source caches from an earlier experiment. Only disposable Kaggle working directories are cleared.

In [5]:
for path in (DATA_CACHE, OUTPUT):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)
print("Fresh initial-training workspace ready.")

Fresh initial-training workspace ready.


## 4. Classical-ML training settings

These settings affect this Kaggle run only. They do not modify the production defaults in the repository.


In [6]:
cpu_count = os.cpu_count() or 4
KAGGLE_ENV = {
    "HF_TOKEN": os.environ.get("HF_TOKEN", ""),
    "HF_HOME": os.environ["HF_HOME"],
    "HF_DATASETS_CACHE": os.environ["HF_DATASETS_CACHE"],
    "TRANSFORMERS_CACHE": os.environ["TRANSFORMERS_CACHE"],
    "MPLBACKEND": "Agg",
    "EXPENSE_ML_DATA_DIR": str(DATA_CACHE),
    "EXPENSE_ML_OUTPUT": str(OUTPUT),
    "EXPENSE_ML_CPU_THREADS": str(min(cpu_count, 8)),
    "EXPENSE_ML_BASELINE_MAX_ROWS": "1500000",
    "EXPENSE_ML_MAX_MERCHANTS": "250000",
    "EXPENSE_ML_DUPLICATE_MAX_ROWS": "500000",
}
print(json.dumps({k: v for k, v in KAGGLE_ENV.items() if k != "HF_TOKEN"}, indent=2))


{
  "HF_HOME": "/kaggle/temp/huggingface",
  "HF_DATASETS_CACHE": "/kaggle/temp/huggingface/datasets",
  "TRANSFORMERS_CACHE": "/kaggle/temp/huggingface/transformers",
  "MPLBACKEND": "Agg",
  "EXPENSE_ML_DATA_DIR": "/kaggle/working/expense-ml-data",
  "EXPENSE_ML_OUTPUT": "/kaggle/working/expense-ml-runs",
  "EXPENSE_ML_CPU_THREADS": "4",
  "EXPENSE_ML_BASELINE_MAX_ROWS": "1500000",
  "EXPENSE_ML_MAX_MERCHANTS": "250000",
  "EXPENSE_ML_DUPLICATE_MAX_ROWS": "500000"
}


## 5. Run the complete initial classical-ML pipeline

The only category classifier trained in this initial run is the **TF-IDF + Logistic Regression** model.

The other applicable components are similarity/retrieval indexes for merchant normalization and duplicate detection. Anomaly detection and spending forecasting run only when the dataset schema contains the required features.

**Transformer training is intentionally disabled for this initial run.**


In [7]:
prepared = DATA_CACHE / "transactions.parquet"
config = ML / "config" / "datasets.yaml"
run(
    UV, "run", "python", "-m", "expense_ml.master_pipeline",
    "--config", str(config),
    "--prepared", str(prepared),
    "--output", str(OUTPUT),
    cwd=ML,
    env=KAGGLE_ENV,
)

$ /usr/local/bin/uv run python -m expense_ml.master_pipeline --config /kaggle/working/expense-tracker/ml/config/datasets.yaml --prepared /kaggle/working/expense-ml-data/transactions.parquet --output /kaggle/working/expense-ml-runs


TF-IDF + Logistic Regression — validation:  18%|█▊        | 2/11 [01:00<04:44, 31.65s/stage]

Removed ambiguous duplicate-text groups before splitting: 2485 groups / 9670 rows



Evaluating tfidf: 100%|██████████| 3746/3746 [00:21<00:00, 177.01batch/s]

Evaluating tfidf-validation-Australia: 100%|██████████| 725/725 [00:03<00:00, 204.97batch/s]

Evaluating tfidf-validation-Canada: 100%|██████████| 729/729 [00:03<00:00, 203.84batch/s]

Evaluating tfidf-validation-India: 100%|██████████| 842/842 [00:06<00:00, 123.69batch/s]

Evaluating tfidf-validation-UK: 100%|██████████| 727/727 [00:03<00:00, 210.30batch/s]

Evaluating tfidf: 100%|██████████| 3751/3751 [00:21<00:00, 177.87batch/s]

Evaluating tfidf-test-Australia: 100%|██████████| 725/725 [00:03<00:00, 201.56batch/s]

Evaluating tfidf-test-Canada: 100%|██████████| 726/726 [00:03<00:00, 207.57batch/s]

Evaluating tfidf-test-India: 100%|██████████| 842/842 [00:06<00:00, 124.52batch/s]

Evaluating tfidf-test-UK: 100%|██████████| 733/733 [00:03<00:00, 209.77batch/s]

Evaluating tfidf-test-USA: 100%|██████████| 727/727 [00:03<00:00, 209.59batch/s]

Reports and export manifest: 100%|██████████| 11/11 [1:28:32<00:00,

{
  "status": "completed",
  "run_id": "20260918T064102Z",
  "pipeline_version": "2.0.0",
  "created_at_utc": "2026-09-18T06:41:02.474685+00:00",
  "python": "3.14.5 (main, May 10 2026, 19:28:16) [Clang 22.1.3 ]",
  "platform": "Linux-6.12.90+-x86_64-with-glibc2.35",
  "seed": 42,
  "taxonomy_categories": [
    "food_dining",
    "transportation",
    "shopping_retail",
    "entertainment_recreation",
    "healthcare_medical",
    "utilities_services",
    "financial_services",
    "income",
    "government_legal",
    "charity_donations"
  ],
  "resources": {
    "cpu_threads": 4,
    "torch_threads": 4,
    "device": "cpu",
    "cuda_devices": 0,
    "torch_configured": false,
    "platform": "Linux-6.12.90+-x86_64-with-glibc2.35"
  },
  "datasets": [],
  "models": {
    "category_tfidf": {
      "status": "trained",
      "artifact": "models/category-tfidf",
      "validation_accuracy": 0.9922900091368398,
      "validation_macro_f1": 0.9924938150790148,
      "final_training_rows":

CompletedProcess(args=['/usr/local/bin/uv', 'run', 'python', '-m', 'expense_ml.master_pipeline', '--config', '/kaggle/working/expense-tracker/ml/config/datasets.yaml', '--prepared', '/kaggle/working/expense-ml-data/transactions.parquet', '--output', '/kaggle/working/expense-ml-runs'], returncode=0)

## 6. Inspect the completed run

The notebook stops with an error if the master pipeline did not finish successfully. The expected initial category model is `category_tfidf`; transformer status should be `disabled`.

In [8]:
run_dirs = sorted(path for path in OUTPUT.iterdir() if path.is_dir() and (path / "manifest.json").exists())
if not run_dirs:
    raise RuntimeError("No master-training run with manifest.json was produced.")

RUN_DIR = run_dirs[-1]
MANIFEST = json.loads((RUN_DIR / "manifest.json").read_text(encoding="utf-8"))
print("Run:", RUN_DIR.name)
print("Status:", MANIFEST.get("status"))
print("Pipeline:", MANIFEST.get("pipeline_version"))
print("\nSelected model:")
print(json.dumps(MANIFEST.get("selected_model"), indent=2))
print("\nFinal test:")
print(json.dumps(MANIFEST.get("test_evaluation"), indent=2))
if MANIFEST.get("india_holdout_evaluation"):
    print("\nIndia holdout:")
    print(json.dumps(MANIFEST["india_holdout_evaluation"], indent=2))
print("\nModel statuses:")
for name, details in MANIFEST.get("models", {}).items():
    print(f"  {name}: {details.get('status')}")

if MANIFEST.get("status") != "completed":
    raise RuntimeError("Master pipeline did not finish with status=completed.")

Run: 20260918T064102Z
Status: completed
Pipeline: 2.0.0

Selected model:
{
  "name": "tfidf",
  "artifact": "models/category-tfidf",
  "selection_metric": "validation_macro_f1",
  "validation_accuracy": 0.9922900091368398,
  "validation_macro_f1": 0.9924938150790148,
  "test_accuracy": 0.9929848825051134,
  "test_macro_f1": 0.9932028965957418
}

Final test:
{
  "model_name": "tfidf",
  "accuracy": 0.9929848825051134,
  "macro_f1": 0.9932028965957418,
  "weighted_f1": 0.9929778167820927,
  "report": {
    "charity_donations": {
      "precision": 0.9999548451187573,
      "recall": 1.0,
      "f1-score": 0.9999774220496264,
      "support": 22145.0
    },
    "entertainment_recreation": {
      "precision": 0.999746589517253,
      "recall": 0.9980183826629564,
      "f1-score": 0.9988817385800189,
      "support": 23718.0
    },
    "financial_services": {
      "precision": 0.9880473946784922,
      "recall": 0.9998247090169682,
      "f1-score": 0.9939011640064125,
      "support": 2

In [9]:
reports = [
    "reports/dataset_summary.json",
    "reports/dataset_quality.json",
    "reports/split_summary.json",
    "reports/training_sampling.json",
    "reports/model_selection_validation.json",
    "reports/model_comparison.json",
    "reports/category_test.json",
    "reports/category_test_country_metrics.json",
    "reports/category_india_holdout.json",
    "reports/duplicate_candidates.json",
    "reports/anomaly_report.json",
    "reports/spending_forecast.json",
]

for relative in reports:
    path = RUN_DIR / relative
    if path.exists():
        print(f"\n===== {relative} =====")
        print(path.read_text(encoding="utf-8")[:12000])


===== reports/dataset_summary.json =====
{
  "rows": 1687124,
  "classes": 10,
  "fingerprint": "d108095d65fb9b6895133560fe3c85d4633ed878c29a4eb8059b20a43d482205",
  "class_counts": {
    "financial_services": 205592,
    "shopping_retail": 202030,
    "transportation": 166379,
    "food_dining": 166367,
    "entertainment_recreation": 165250,
    "utilities_services": 159787,
    "healthcare_medical": 159555,
    "government_legal": 156292,
    "charity_donations": 153635,
    "income": 152237
  },
  "country_counts": {
    "India": 439581,
    "Canada": 312426,
    "UK": 312032,
    "Australia": 311926,
    "USA": 311159
  },
  "source_counts": {
    "global-transaction-categorization": 1560218,
    "finee-india": 126175,
    "synthetic-indian-transactions": 731
  }
}

===== reports/dataset_quality.json =====
{
  "rows": 1687124,
  "columns": [
    "text",
    "label",
    "source",
    "source_label",
    "country",
    "currency",
    "language",
    "record_id"
  ],
  "missing_by

## 7. Package the initial model artifacts

The archive contains the complete master run: trained models, manifests, reports, and figures. Raw datasets and Hugging Face caches are not included.

In [10]:
archive_base = WORK_ROOT / f"expense-tracker-ml-initial-{RUN_DIR.name}"
archive_path = Path(shutil.make_archive(str(archive_base), "zip", root_dir=RUN_DIR))
print("Kaggle artifact:", archive_path)
print("Artifact size MiB:", round(archive_path.stat().st_size / 1024**2, 2))
print("Run directory:", RUN_DIR)

Kaggle artifact: /kaggle/working/expense-tracker-ml-initial-20260918T064102Z.zip
Artifact size MiB: 290.27
Run directory: /kaggle/working/expense-ml-runs/20260918T064102Z
